# 02. Domain-Specific English LM Adaptation (Agro-Extension Corpus)

**Course**: ICS554 Natural Language Processing · Ashesi University  
**Team**: MICS 2028 · Group 1  
**Project**: Prosit 1 (Ankora AI Research Lab)  
**Domain Corpus**: Real Agricultural Advisory Dataset (`KisanVaani/agriculture-qa-english-only`)  
**Methodology**: Parameter-Efficient Fine-Tuning (PEFT / LoRA) targeting attention projections (`Conv1D`).

---
### Pipeline Overview
1. **Domain Corpus Preparation**: Real extension Q&A pairs split into train, val, and test.
2. **Base Foundation Model Evaluation**: Measure zero-shot domain perplexity baseline on `distilgpt2`.
3. **LoRA Adapter Configuration**: Inject low-rank matrices ($r=8, \alpha=32$) into attention projections to slash compute and prevent catastrophic forgetting.
4. **Fine-Tuning Execution**: Train causal LM on domain text.
5. **Post-Adaptation Benchmark**: Quantify perplexity reduction and audit domain prompt completions.

In [ ]:
import sys
from pathlib import Path
import torch
import math

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import matplotlib.pyplot as plt
import seaborn as sns

print("PyTorch version:", torch.__version__)
print("Environment ready!")

## 1. Load Real Agricultural Domain Splits
We load real agricultural extension text prepared in .

In [ ]:
PROCESSED_DIR = REPO_ROOT / "data" / "processed" / "domain_english"

def load_split(path, max_samples=None):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    examples = [e.strip() for e in text.split("

") if e.strip()]
    return examples[:max_samples] if max_samples else examples

train_texts = load_split(PROCESSED_DIR / "train.txt", max_samples=500)
val_texts = load_split(PROCESSED_DIR / "val.txt", max_samples=100)
test_texts = load_split(PROCESSED_DIR / "test.txt", max_samples=100)

print(f"Loaded splits: Train={len(train_texts)}, Val={len(val_texts)}, Test={len(test_texts)}")
print("
Sample record:")
print(train_texts[0])

## 2. Load Base Model & Baseline Perplexity
We load  and evaluate its baseline confusion (perplexity) on held-out agricultural questions before training.

In [ ]:
BASE_MODEL_NAME = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME)
base_model.eval()

def evaluate_ppl(model, tokenizer, texts, batch_size=8, max_length=96):
    model.eval()
    total_loss, total_batches = 0.0, 0
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = tokenizer(batch_texts, truncation=True, max_length=max_length, padding=True, return_tensors="pt")
        labels = enc.input_ids.clone()
        labels[labels == tokenizer.pad_token_id] = -100
        with torch.no_grad():
            loss = model(input_ids=enc.input_ids, attention_mask=enc.attention_mask, labels=labels).loss.item()
            total_loss += loss
            total_batches += 1
    avg_loss = total_loss / max(total_batches, 1)
    return avg_loss, math.exp(avg_loss)

base_loss, base_ppl = evaluate_ppl(base_model, tokenizer, test_texts)
print(f"Base Model Zero-Shot Loss: {base_loss:.4f}")
print(f"Base Model Zero-Shot Perplexity on Agricultural Test Set: {base_ppl:.2f}")

### Qualitative Test: Asking the Untrained Base Model a Farming Question

In [ ]:
prompt = "Question: why is crop rotation important in farming?
Answer:"
input_ids = tokenizer.encode(prompt, return_tensors="pt")
with torch.no_grad():
    out = base_model.generate(input_ids, max_new_tokens=40, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## 3. Configure LoRA (Low-Rank Adaptation)
We freeze 99.82% of the model and only train low-rank matrices ($r=8, \alpha=32$) on attention projections.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    fan_in_fan_out=True,
)
adapted_model = get_peft_model(base_model, lora_config)
adapted_model.print_trainable_parameters()

## 4. Fine-Tune with Hugging Face Trainer
We train the LoRA adapter on our domain training set.

In [ ]:
def tokenize_func(batch):
    return tokenizer(batch["text"], truncation=True, max_length=96, padding="max_length")

train_ds = Dataset.from_dict({"text": train_texts}).map(tokenize_func, batched=True, remove_columns=["text"])
val_ds = Dataset.from_dict({"text": val_texts}).map(tokenize_func, batched=True, remove_columns=["text"])

training_args = TrainingArguments(
    output_dir=str(REPO_ROOT / "models" / "temp_nb_train"),
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=5e-4,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    use_cpu=True,
)

trainer = Trainer(
    model=adapted_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("Starting LoRA fine-tuning...")
trainer.train()
print("Training complete!")

## 5. Post-Adaptation Perplexity & Generation Comparison
We measure perplexity on the unseen test set and prompt the adapted model.

In [ ]:
adapt_loss, adapt_ppl = evaluate_ppl(adapted_model, tokenizer, test_texts)
print(f"Adapted Model Test Loss: {adapt_loss:.4f}")
print(f"Adapted Model Test Perplexity: {adapt_ppl:.2f}")
print(f"Perplexity Improvement: {base_ppl - adapt_ppl:.2f} points ({(base_ppl - adapt_ppl)/base_ppl*100:.1f}% reduction)")

# Plot comparison
plt.figure(figsize=(7, 4))
sns.set_theme(style="whitegrid")
bars = plt.bar(["Base Model (Zero-Shot)", "LoRA Adapted ($r=8$)"], [base_ppl, adapt_ppl], color=["#4A90E2", "#50E3C2"], width=0.5)
for b, p in zip(bars, [base_ppl, adapt_ppl]):
    plt.text(b.get_x() + b.get_width()/2, p + 1, f"{p:.1f}", ha="center", fontweight="bold")
plt.ylabel("Perplexity (Lower is Better)")
plt.title("Agro-Extension Domain Adaptation Benchmark")
plt.show()

### Qualitative Prompt Re-Test

In [ ]:
with torch.no_grad():
    out_adapt = adapted_model.generate(input_ids, max_new_tokens=40, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
print("--- Prompt ---")
print(prompt)
print("
--- Adapted Model Completion ---")
print(tokenizer.decode(out_adapt[0], skip_special_tokens=True))

## 6. Decoding Ablation: Eliminating Repetitive Loops

Small language models suffer from self-reinforcement bias, where generated tokens increase the probability of repeating themselves.
We benchmark five decoding strategies using repetition penalties ($r=1.25\dots 1.3$), $N$-gram blocking ($N=3$), and temperature tuning ($T=0.35$).

In [ ]:
# Benchmark decoding strategies on domain prompts
decoding_configs = {
    "Unpenalized Baseline": {"temperature": 0.7, "top_p": 0.9, "repetition_penalty": 1.0, "do_sample": True},
    "Repetition Penalty (r=1.3)": {"temperature": 0.7, "top_p": 0.9, "repetition_penalty": 1.3, "do_sample": True},
    "N-gram Blocking (N=3)": {"temperature": 0.7, "top_p": 0.9, "no_repeat_ngram_size": 3, "do_sample": True},
    "Conservative Agronomic (T=0.35)": {"temperature": 0.35, "top_p": 0.85, "repetition_penalty": 1.25, "no_repeat_ngram_size": 3, "do_sample": True},
}

test_prompt = "Question: How can farmers control fall armyworm in maize?\nAnswer:"
input_ids = tokenizer.encode(test_prompt, return_tensors="pt")

print(f"PROMPT: {test_prompt}\n" + "="*50)
for name, kwargs in decoding_configs.items():
    with torch.no_grad():
        out = adapted_model.generate(input_ids, max_new_tokens=40, pad_token_id=tokenizer.eos_token_id, **kwargs)
    ans = tokenizer.decode(out[0], skip_special_tokens=True)[len(test_prompt):].strip()
    print(f"\n[{name}]:\n-> {ans}")


## 7. Training Improvement: Prompt Loss Masking

In standard causal language modeling, loss is computed on both prompt and answer tokens.
With **Prompt Loss Masking**, we set `labels[:prompt_len] = -100`, forcing 100% of gradient updates to optimize answer generation conditioned on the question.

In [ ]:
# Demonstration of prompt-loss masking logic
def mask_prompt_labels(example, tokenizer, max_length=96):
    parts = example.split("\nAnswer:")
    prompt_len = len(tokenizer.encode(parts[0] + "\nAnswer:", add_special_tokens=False)) if len(parts) == 2 else 0
    enc = tokenizer(example, truncation=True, max_length=max_length, padding="max_length", return_tensors="pt")
    labels = enc.input_ids.clone()
    labels[0, :prompt_len] = -100  # PyTorch CrossEntropyLoss ignores -100
    labels[labels == tokenizer.pad_token_id] = -100
    return enc.input_ids, labels

sample_ids, sample_labels = mask_prompt_labels(train_texts[0], tokenizer)
print("Input IDs shape:", sample_ids.shape)
print("Masked tokens count (-100):", (sample_labels == -100).sum().item())
print("Active learning tokens count:", (sample_labels != -100).sum().item())
